# Multi-Criteria Decision Analysis (MCDA)

MCDA on rasters treats each pixel as a decision alternative, each raster layer as a criterion,
and produces a single composite surface where every pixel gets a score.
Common applications: site suitability, hazard mapping, conservation prioritization.

The workflow has four stages:

```
raw criteria -> standardize (0-1) -> weight -> combine -> suitability surface
                                                  |
                                            constraints -> mask exclusion zones
                                                  |
                                            sensitivity -> stability check
```

This notebook walks through each stage using synthetic terrain data.

In [ ]:
%matplotlib inline
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from xrspatial import generate_terrain, slope, aspect
from xrspatial.proximity import proximity
from xrspatial.mcda import (
    standardize, ahp_weights, rank_weights,
    wlc, wpm, owa, fuzzy_overlay, boolean_overlay,
    constrain, sensitivity,
)

## Generate synthetic criterion layers

We'll create a terrain surface and derive slope, aspect, and a
synthetic "distance to road" layer from it.

In [ ]:
canvas = xr.DataArray(np.zeros((300, 300)), dims=['y', 'x'])
terrain = generate_terrain(canvas=canvas)

slp = slope(terrain)
asp = aspect(terrain)

# Synthetic distance-to-road: place a few "road" pixels and compute proximity
rng = np.random.default_rng(1030)
road_mask = np.zeros((300, 300), dtype=np.float64)
road_mask[150, :] = 1  # horizontal road
road_mask[:, 100] = 1  # vertical road
road_raster = xr.DataArray(road_mask, dims=['y', 'x'])
dist_road = proximity(road_raster)

# Water mask for constraints
water = terrain < float(np.nanpercentile(terrain.values, 10))

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
terrain.plot.imshow(ax=axes[0, 0], cmap='terrain', add_colorbar=True)
axes[0, 0].set_title('Elevation')
axes[0, 0].set_axis_off()

slp.plot.imshow(ax=axes[0, 1], cmap='YlOrRd', add_colorbar=True)
axes[0, 1].set_title('Slope (degrees)')
axes[0, 1].set_axis_off()

dist_road.plot.imshow(ax=axes[1, 0], cmap='Blues_r', add_colorbar=True)
axes[1, 0].set_title('Distance to road')
axes[1, 0].set_axis_off()

water.plot.imshow(ax=axes[1, 1], cmap='Blues', add_colorbar=False)
axes[1, 1].set_title('Water mask (exclusion zone)')
axes[1, 1].set_axis_off()

plt.tight_layout()

## Stage 1: Standardize criteria to 0-1

Each criterion uses a different value function depending on how
its raw values relate to suitability:

- **Slope**: lower is better (linear)
- **Distance to road**: closer is better (sigmoidal decay)
- **Aspect**: south-facing (180 deg) is ideal (gaussian)

In [ ]:
slope_std = standardize(slp, method='linear',
                        direction='lower_is_better', bounds=(0, 45))

dist_std = standardize(dist_road, method='sigmoidal',
                       midpoint=80, spread=-0.05)

aspect_std = standardize(asp, method='gaussian',
                         mean=180, std=60)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, data, title in zip(axes,
                           [slope_std, dist_std, aspect_std],
                           ['Slope (linear, lower better)',
                            'Distance (sigmoidal decay)',
                            'Aspect (gaussian, south ideal)']):
    data.plot.imshow(ax=ax, cmap='RdYlGn', vmin=0, vmax=1,
                     add_colorbar=True)
    ax.set_title(title)
    ax.set_axis_off()
plt.tight_layout()

### Other standardization methods

The `standardize` function supports six methods. Here are the
remaining three applied to elevation:

In [ ]:
elev_min = float(np.nanmin(terrain.values))
elev_max = float(np.nanmax(terrain.values))
elev_mid = (elev_min + elev_max) / 2

tri_std = standardize(terrain, method='triangular',
                      low=elev_min, peak=elev_mid, high=elev_max)

pw_std = standardize(terrain, method='piecewise',
                     breakpoints=[elev_min, elev_mid * 0.5,
                                  elev_mid, elev_mid * 1.5, elev_max],
                     values=[0.0, 0.3, 1.0, 0.6, 0.0])

# Categorical: bin elevation into 4 classes
bins = np.linspace(elev_min, elev_max, 5)
elev_class = np.digitize(terrain.values, bins).astype(np.float64)
elev_class_da = xr.DataArray(elev_class, dims=['y', 'x'])
cat_std = standardize(elev_class_da, method='categorical',
                      mapping={1: 0.9, 2: 0.7, 3: 0.3, 4: 0.1, 5: 0.0})

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, data, title in zip(axes,
                           [tri_std, pw_std, cat_std],
                           ['Triangular', 'Piecewise', 'Categorical']):
    data.plot.imshow(ax=ax, cmap='RdYlGn', vmin=0, vmax=1,
                     add_colorbar=True)
    ax.set_title(title)
    ax.set_axis_off()
plt.tight_layout()

## Stage 2: Derive weights

### AHP (Analytical Hierarchy Process)

Provide pairwise comparisons on a 1-9 scale. AHP computes weights
via the principal eigenvector and flags inconsistent judgments
(consistency ratio > 0.10).

In [ ]:
weights_ahp, consistency = ahp_weights(
    criteria=['slope', 'distance', 'aspect'],
    comparisons={
        ('slope', 'distance'): 3,     # slope 3x more important
        ('slope', 'aspect'): 5,       # slope 5x more important
        ('distance', 'aspect'): 2,    # distance 2x more important
    }
)

print('AHP weights:')
for k, v in weights_ahp.items():
    print(f'  {k}: {v:.3f}')
print(f'\nConsistency ratio: {consistency.ratio:.4f}')
print(f'Consistent (CR < 0.10): {consistency.is_consistent}')

### Rank-order weights

When you can rank criteria but can't quantify pairwise comparisons,
use `rank_weights` with ROC (rank-order centroid), rank sum, or
reciprocal methods.

In [ ]:
for method in ['roc', 'rs', 'rr']:
    w = rank_weights(['slope', 'distance', 'aspect'], method=method)
    print(f'{method}: ' + ', '.join(f'{k}={v:.3f}' for k, v in w.items()))

## Stage 3: Combine layers

Bundle standardized criteria into an `xr.Dataset` and apply
different combination methods.

In [ ]:
criteria = xr.Dataset({
    'slope': slope_std,
    'distance': dist_std,
    'aspect': aspect_std,
})

# Use the AHP-derived weights
suit_wlc = wlc(criteria, weights_ahp)
suit_wpm = wpm(criteria, weights_ahp)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
suit_wlc.plot.imshow(ax=axes[0], cmap='RdYlGn', vmin=0, vmax=1,
                     add_colorbar=True)
axes[0].set_title('WLC (Weighted Linear Combination)')
axes[0].set_axis_off()

suit_wpm.plot.imshow(ax=axes[1], cmap='RdYlGn', vmin=0, vmax=1,
                     add_colorbar=True)
axes[1].set_title('WPM (Weighted Product Model)')
axes[1].set_axis_off()

plt.tight_layout()

### OWA: risk attitude via order weights

OWA adds a second set of weights applied by rank position at each pixel.
Weighting the lowest-scoring criterion produces conservative (AND-like)
results; weighting the highest produces optimistic (OR-like) results.

In [ ]:
# Conservative: worst criterion matters most
suit_conservative = owa(criteria, weights_ahp,
                        order_weights=[0.1, 0.3, 0.6])

# Optimistic: best criterion matters most
suit_optimistic = owa(criteria, weights_ahp,
                      order_weights=[0.6, 0.3, 0.1])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
suit_conservative.plot.imshow(ax=axes[0], cmap='RdYlGn', vmin=0, vmax=1,
                              add_colorbar=True)
axes[0].set_title('OWA conservative (risk-averse)')
axes[0].set_axis_off()

suit_optimistic.plot.imshow(ax=axes[1], cmap='RdYlGn', vmin=0, vmax=1,
                             add_colorbar=True)
axes[1].set_title('OWA optimistic (risk-tolerant)')
axes[1].set_axis_off()

plt.tight_layout()

### Fuzzy overlay

No explicit weights. Combine using fuzzy set operators.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for ax, op, title in zip(axes.flat,
                         ['and', 'or', 'product', 'sum', 'gamma'],
                         ['AND (min)', 'OR (max)', 'Product',
                          'Algebraic Sum', 'Gamma (0.9)']):
    kw = {'gamma': 0.9} if op == 'gamma' else {}
    result = fuzzy_overlay(criteria, operator=op, **kw)
    result.plot.imshow(ax=ax, cmap='RdYlGn', vmin=0, vmax=1,
                       add_colorbar=True)
    ax.set_title(title)
    ax.set_axis_off()

axes[1, 2].set_visible(False)
plt.tight_layout()

### Boolean overlay

Binary suitability from hard thresholds.

In [ ]:
suitable = boolean_overlay({
    'gentle_slope': slp < 15,
    'near_road': dist_road < 80,
    'not_water': ~water,
}, operator='and')

fig, ax = plt.subplots(figsize=(6, 5))
suitable.astype(float).plot.imshow(ax=ax, cmap='Greens',
                                   add_colorbar=False)
ax.set_title('Boolean overlay (slope<15 AND near road AND not water)')
ax.set_axis_off()
plt.tight_layout()

## Stage 4: Constraints and sensitivity

### Apply constraints

Mask out water areas from the WLC suitability surface.

In [ ]:
suit_constrained = constrain(suit_wlc, exclude=[water])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
suit_wlc.plot.imshow(ax=axes[0], cmap='RdYlGn', vmin=0, vmax=1,
                     add_colorbar=True)
axes[0].set_title('Before constraints')
axes[0].set_axis_off()

suit_constrained.plot.imshow(ax=axes[1], cmap='RdYlGn', vmin=0, vmax=1,
                             add_colorbar=True)
axes[1].set_title('After masking water')
axes[1].set_axis_off()

plt.tight_layout()

### Sensitivity analysis

Check whether the results change when weights shift by a few percent.
High sensitivity means the output depends more on weight choices
than on spatial data.

In [ ]:
# One-at-a-time: how much does each criterion's weight perturbation affect scores?
sens_oat = sensitivity(criteria, weights_ahp,
                       method='one_at_a_time', delta=0.05)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, var in zip(axes, sens_oat.data_vars):
    sens_oat[var].plot.imshow(ax=ax, cmap='YlOrRd', add_colorbar=True)
    ax.set_title(f'Sensitivity: {var}')
    ax.set_axis_off()
plt.tight_layout()

In [ ]:
# Monte Carlo: overall stability across random weight vectors
sens_mc = sensitivity(criteria, weights_ahp,
                      method='monte_carlo', n_samples=200)

fig, ax = plt.subplots(figsize=(6, 5))
sens_mc.plot.imshow(ax=ax, cmap='YlOrRd', add_colorbar=True,
                    cbar_kwargs={'label': 'Coefficient of variation'})
ax.set_title('Monte Carlo sensitivity (200 samples)')
ax.set_axis_off()
plt.tight_layout()

print(f'Mean CV: {float(sens_mc.mean()):.4f}')
print(f'Max CV:  {float(sens_mc.max()):.4f}')

## Comparison of combination methods

Side-by-side view of all methods on the same criteria and weights.

In [ ]:
results = {
    'WLC': suit_wlc,
    'WPM': suit_wpm,
    'OWA (conservative)': suit_conservative,
    'OWA (optimistic)': suit_optimistic,
    'Fuzzy AND': fuzzy_overlay(criteria, operator='and'),
    'Fuzzy Gamma 0.9': fuzzy_overlay(criteria, operator='gamma', gamma=0.9),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (title, data) in zip(axes.flat, results.items()):
    data.plot.imshow(ax=ax, cmap='RdYlGn', vmin=0, vmax=1,
                     add_colorbar=True)
    ax.set_title(title)
    ax.set_axis_off()
plt.tight_layout()